# Mesh 2d

Reusable function definitions only. Original project inputs, run settings and saved outputs are not included. Running these cells defines functions; it does not read a dataset. Start with the [synthetic walkthrough](../examples/synthetic_walkthrough.ipynb) for a complete small run.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "helper" / "pipeline_utils.py").is_file()
             and (p / "examples" / "synthetic_demo.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook inside this repository.")
for folder in (ROOT, ROOT / "Invert-PropOnly-saveCSV", ROOT / "Invert-Prop+cond-saveCSV"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))


In [ ]:
#!/usr/bin/env python3
"""
invert_two_outputs_mesh_select_Q.py

Mesh‐grid + Newton inversion for two‐output neural networks under the
constraint x1_raw = Q * x0_raw. Returns up to k+1 solutions:
  (adj_inputs_raw (4,), preds_raw (2,), resid_raw (2,))
with the first entry always the “actual” test‐data point.

This version is concise, well‐commented, and reuses shared utilities.
"""

import numpy as np
import torch

from invert_two_utils import (
    initialize_two_output_inversion,
    fitness_two_outputs,
    select_top_k_unique,
    unpack_two_output_solutions
)


def _generate_mesh_cells(
    min_vec: np.ndarray,
    max_vec: np.ndarray,
    mesh_dims: tuple[int, int, int, int]
) -> list[tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """
    Partition the 4D search hyperrectangle [min_vec, max_vec] into a regular
    grid of mesh_dims cells. Returns list of (center, cell_min, cell_max).
    """
    mesh = np.array(mesh_dims, int)
    steps = (max_vec - min_vec) / mesh
    cells = []
    for idx in np.ndindex(*mesh):
        idx = np.array(idx)
        cmin   = min_vec + idx * steps
        cmax   = cmin + steps
        center = cmin + 0.5 * steps
        cells.append((center, cmin, cmax))
    return cells


def _refine_newton(
    center: np.ndarray,
    cmin: np.ndarray,
    cmax: np.ndarray,
    model1: torch.nn.Module,
    model2: torch.nn.Module,
    x_fixed_s: np.ndarray,
    means_x: np.ndarray,
    scales_x: np.ndarray,
    target_s: np.ndarray,
    MAX_ITER: int,
    STEP_SIZE: float,
    TOL: float
) -> np.ndarray:
    """
    From the initial “center” in scaled space, run Newton–Raphson with Gauss–
    Newton steps (clipped to [cmin,cmax]) to reduce the combined scaled
    residual of the two outputs below TOL. Returns the final scaled vector.
    """
    # Initialize a trainable tensor for the adjustable variables
    x_var = torch.tensor(center, dtype=torch.float32, requires_grad=True)
    low   = torch.from_numpy(cmin).float()
    high  = torch.from_numpy(cmax).float()

    for _ in range(MAX_ITER):
        if x_var.grad is not None:
            x_var.grad.zero_()

        # Unpack scaled variables
        x0_s, x2_s, x3_s, Q = x_var
        # Reconstruct x1_s via Q * raw x0
        x0_raw = x0_s * scales_x[0] + means_x[0]
        x1_raw = Q    * x0_raw
        x1_s   = (x1_raw - means_x[1]) / scales_x[1]

        # Build full input tensor (requires_grad)
        inp = torch.cat([
            torch.stack([x0_s, x1_s, x2_s, x3_s]),
            torch.from_numpy(x_fixed_s).float()
        ])

        # Forward pass with gradient tracking
        out1 = model1(inp)[0]
        out2 = model2(inp)[1]

        # Compute scaled errors
        err1 = out1 - target_s[0]
        err2 = out2 - target_s[1]
        if torch.hypot(err1.detach(), err2.detach()).item() < TOL:
            break

        # Compute Gauss–Newton step: JᵀJ Δ = Jᵀ e
        g1 = torch.autograd.grad(err1, x_var, retain_graph=True)[0]
        g2 = torch.autograd.grad(err2, x_var)[0]
        JTJ = g1.unsqueeze(1) @ g1.unsqueeze(0) + g2.unsqueeze(1) @ g2.unsqueeze(0)
        JTe = g1 * err1.detach() + g2 * err2.detach()
        try:
            delta = torch.linalg.solve(JTJ, JTe)
        except RuntimeError:
            break

        # Update and clip into the current cell
        x_var = (x_var - STEP_SIZE * delta).clamp(min=low, max=high).detach()
        x_var.requires_grad_(True)

    return x_var.detach().cpu().numpy()


def invert_two_outputs_mesh_select_Q(
    x_scaler_path: str,
    y_scaler_path: str,
    model1_path: str,
    model2_path: str,
    x_fixed_raw: np.ndarray,
    actual_x_adj_raw: np.ndarray,
    target1_raw: float,
    target2_raw: float,
    mesh_dims: tuple[int, int, int, int],
    Q_bounds: tuple[float, float],
    k: int,
    q: int,
    MAX_ITER: int,
    STEP_SIZE: float,
    TOL: float,
    hidden_dims1: list[int],
    hidden_dims2: list[int],
    activation: str,
    x_adj_intervals_raw: np.ndarray
) -> list[tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """
    Main inversion routine:
      1. Initialize scalers, models, scaled inputs, “actual” result, and search bounds.
      2. Generate mesh cells over [x0_s, x2_s, x3_s, Q_raw].
      3. Score each cell center by summed squared scaled errors.
      4. Select top (k*q) cells and refine via Newton.
      5. From all refinements, pick top-k unique.
      6. Inverse-scale to raw adjustable inputs, raw predictions, and residuals.
      7. Return list with the first tuple = actual test‐data point.
    """
    # Shared initialization from invert_two_utils
    init = initialize_two_output_inversion(
        x_scaler_path, y_scaler_path,
        model1_path, model2_path,
        hidden_dims1, hidden_dims2,
        activation,
        x_fixed_raw, actual_x_adj_raw,
        target1_raw, target2_raw,
        x_adj_intervals_raw, Q_bounds
    )
    model1, model2 = init["model1"], init["model2"]
    means_x, scales_x = init["means_x"], init["scales_x"]
    scales_y, mean_y  = init["scales_y"], init["mean_y"]
    x_fixed_s        = init["x_fixed_s"]
    results          = init["results"]      # actual row
    min_vec, max_vec = init["min_vec"], init["max_vec"]
    target_s         = init["target_s"]     # scaled [t1, t2]

    # 2. Build mesh cells
    cells = _generate_mesh_cells(min_vec, max_vec, mesh_dims)

    # 3. Score each center
    scored = []
    for center, cmin, cmax in cells:
        score = fitness_two_outputs(
            center, model1, model2,
            x_fixed_s, means_x, scales_x,
            target_s
        )
        scored.append((score, center, cmin, cmax))
    scored.sort(key=lambda x: x[0])

    # 4. Refine top (k*q) cells
    selected = scored[: k * q]
    refined = [
        _refine_newton(center, cmin, cmax,
                       model1, model2, x_fixed_s,
                       means_x, scales_x, target_s,
                       MAX_ITER, STEP_SIZE, TOL)
        for _, center, cmin, cmax in selected
    ]

    # 5. Pick top-k unique refined candidates
    fitnesses   = [
        fitness_two_outputs(c, model1, model2, x_fixed_s, means_x, scales_x, target_s)
        for c in refined
    ]
    best_scaled = select_top_k_unique(fitnesses, refined, k)

    # 6. Inverse-scale and compute raw preds/residuals
    solutions = unpack_two_output_solutions(
        best_scaled,
        model1, model2,
        means_x, scales_x,
        x_fixed_s,
        scales_y, mean_y,
        target1_raw, target2_raw
    )

    # 7. Return actual row + solutions
    return results + solutions
